# Demo 2 - Ray Job via the CodeFlare SDK (from a workbench)

The CodeFlare SDK submits a `RayJob` that spins up an ephemeral Ray cluster,
runs the entrypoint on GPU workers, and tears it down. `local_queue` routes it
through Kueue. In a workbench the SDK auto-authenticates in-cluster.

In [4]:
!pip install codeflare-sdk

Looking in indexes: https://packages.redhat.com/api/pypi/public-rhai/rhoai/3.5/cpu-ubi9/simple/
   | 263.9 kB 902.0 kB/s 0:00:00
   \ 141.1 kB 1.2 MB/s 0:00:00
   - 76.9 kB 122.3 MB/s 0:00:00
   / 73.8 MB 18.6 MB/s 0:00:04
   | 916.8 kB 4.3 MB/s 0:00:00
   - 396.0 kB 19.8 MB/s 0:00:00
   - 312.1 kB 22.3 MB/s 0:00:00
   \ 2.2 MB 22.0 MB/s 0:00:00
   - 33.9 kB 31.9 MB/s 0:00:00
   \ 2.1 MB 17.4 MB/s 0:00:00
   - 32.9 kB 4.0 MB/s 0:00:00
   / 5.5 MB 19.8 MB/s 0:00:00
   - 472.3 kB 23.5 MB/s 0:00:00
   - 35.9 kB 55.6 MB/s 0:00:00
   - 203.0 kB 34.4 MB/s 0:00:00
   - 125.8 kB 74.0 MB/s 0:00:00
   - 175.9 kB 9.5 MB/s 0:00:00
   - 52.1 kB 11.9 MB/s 0:00:00
   - 210.6 kB 16.9 MB/s 0:00:00
   - 269.8 kB 16.9 MB/s 0:00:00
   - 162.6 kB 16.8 MB/s 0:00:00
   - 1.1 MB 16.9 MB/s 0:00:00
   - 75.2 kB 207.6 MB/s 0:00:00
  Attempting uninstall: rich╺━━━━━━━━━━━━━━━━━━━ 13/26 [virtualenv]
    Found existing installation: rich 15.0.0━━━━━━━━━━━━━━━━━━ 13/26 [virtualenv]
    Uninstalling rich-15.0.0:╺━━━━

In [47]:
# --- shared helpers: run once at the top of the notebook ---------------------
import time
from kubernetes import client, config

# Inside an OpenShift AI workbench we're authenticated via the mounted service
# account, so in-cluster config just works -- no token, no login.
config.load_incluster_config()

def current_namespace(default="gpu-aas-demo"):
    """The workbench's own project namespace (where your LocalQueue must live)."""
    try:
        return open("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read().strip()
    except FileNotFoundError:
        return default

NAMESPACE = current_namespace()
LOCAL_QUEUE = "gpu-local-queue"
co = client.CustomObjectsApi()
print("Submitting into namespace:", NAMESPACE, "| LocalQueue:", LOCAL_QUEUE)

def check_local_queue():
    try:
        lq = co.get_namespaced_custom_object(
            "kueue.x-k8s.io", "v1beta1", NAMESPACE, "localqueues", LOCAL_QUEUE)
        print("LocalQueue OK -> clusterQueue:", lq["spec"]["clusterQueue"])
    except client.ApiException as e:
        print(f"[!] LocalQueue '{LOCAL_QUEUE}' not found in '{NAMESPACE}' (HTTP {e.status}).")
        print("    Run the workbench INSIDE the gpu-aas-demo project, or create a")
        print("    LocalQueue in this namespace pointing at gpu-cluster-queue.")

def print_workloads():
    """Show Kueue's view: which workloads are admitted vs waiting on quota."""
    items = co.list_namespaced_custom_object(
        "kueue.x-k8s.io", "v1beta1", NAMESPACE, "workloads").get("items", [])
    if not items:
        print("  (no workloads yet)")
    for w in items:
        conds = {c["type"]: c["status"] for c in w.get("status", {}).get("conditions", [])}
        state = "ADMITTED" if conds.get("Admitted") == "True" else \
                ("QuotaReserved" if conds.get("QuotaReserved") == "True" else "waiting")
        print(f"  {w['metadata']['name']:45s} {state}")

Submitting into namespace: gpu-aas-demo-1 | LocalQueue: gpu-local-queue


In [48]:
check_local_queue()

LocalQueue OK -> clusterQueue: gpu-cluster-queue


In [49]:
from codeflare_sdk import RayJob, ManagedClusterConfig

# Ephemeral Ray cluster shape. worker_extended_resource_requests is how you ask
# for GPUs; the old num_gpus / min_* / max_* params are deprecated.
cluster_config = ManagedClusterConfig(
    head_cpu_requests="1", head_cpu_limits="2",
    head_memory_requests=4, head_memory_limits=8,
    head_accelerators={"nvidia.com/gpu": 0},     # head needs no GPU
    num_workers=2,
    worker_cpu_requests="2", worker_cpu_limits="4",
    worker_memory_requests=8, worker_memory_limits=16,
    worker_accelerators={"nvidia.com/gpu": 1},   # <-- GPUs go here, 2 workers x 1 = 2 GPUs
)

ENTRYPOINT = (
    "python -c "
    "'import ray; ray.init(); "
    "print(\"Ray resources:\", ray.cluster_resources()); "
    "g = ray.remote(num_gpus=1)(lambda: ray.get_gpu_ids()); "
    "print(\"GPU ids on workers:\", ray.get([g.remote() for _ in range(2)]))'"
)

job = RayJob(
    job_name="ray-training-demo",
    entrypoint=ENTRYPOINT,
    cluster_config=cluster_config,
    namespace=NAMESPACE,
    local_queue=LOCAL_QUEUE,
)

try:
    client.CustomObjectsApi().delete_namespaced_custom_object(
        group="ray.io", version="v1", namespace=NAMESPACE,
        plural="rayjobs", name="ray-training-demo")
    print("deleted previous RayJob, waiting for cleanup...")
    time.sleep(5)
except client.ApiException as e:
    if e.status != 404:
        raise   # 404 = nothing to delete, which is fine

job.submit()
print("Submitted RayJob:", job.name)

2026-09-14 13:44:07,587 Creating new cluster: ray-training-demo-cluster
2026-09-14 13:44:07,590 Initialized RayJob: ray-training-demo in namespace: gpu-aas-demo-1


deleted previous RayJob, waiting for cleanup...


2026-09-14 13:44:12,606 Built RayCluster spec using RayJob-specific builder for cluster: ray-training-demo-cluster
2026-09-14 13:44:12,607 RayJob will create new cluster: ray-training-demo-cluster
2026-09-14 13:44:12,607 Submitting RayJob ray-training-demo to Kuberay operator
2026-09-14 13:44:12,623 Successfully submitted RayJob ray-training-demo


Submitted RayJob: ray-training-demo


### Admission + status
Re-run to watch it get admitted (2 GPUs). If the batch job is holding a GPU, you'll see this wait.

In [50]:
import time
from IPython.display import clear_output

for i in range(60):  # ~10 min max; interrupt the kernel (■) to stop early
    clear_output(wait=True)
    print(f"=== refresh {i+1}  (t+{i*10}s) ===")
    print_workloads()
    try:
        status, ready = job.status()
        print("\nRayJob status:", status)
    except Exception as e:
        status = None
        print("\nstatus not ready yet:", e)

    # stop once the job reaches a terminal state
    if status is not None and str(status).split(".")[-1].rstrip(":>") in ("COMPLETE", "COMPLETED", "FAILED"):
        print("\nDone.")
        break
    time.sleep(10)

=== refresh 6  (t+50s) ===
  rayjob-ray-training-demo-b98f8                ADMITTED


               📦 CodeFlare RayJob Status 📦              
                                                          
 ╭──────────────────────────────────────────────────────╮ 
 │  Name                                                │ 
 │  ray-training-demo                      Complete ✅  │ 
 │                                                      │ 
 │  Job ID: ray-training-demo-ld2rd                     │ 
 │  Status: Complete                                    │ 
 │  RayCluster: ray-training-demo-cluster               │ 
 │  Namespace: gpu-aas-demo-1                           │ 
 │                                                      │ 
 │  Started: 2026-09-14T13:44:12Z                       │ 
 ╰──────────────────────────────────────────────────────╯


RayJob status: CodeflareRayJobStatus.COMPLETE

Done.


### Cleanup

In [51]:
try:
    job.stop()
    print("Stopped/cleaned up RayJob")
except Exception as e:
    print("Nothing to stop:", e)

2026-09-14 13:45:11,165 Successfully suspended rayjob ray-training-demo in namespace gpu-aas-demo-1
2026-09-14 13:45:11,166 Successfully stopped the RayJob ray-training-demo


Stopped/cleaned up RayJob
